In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit

In [ ]:
TASK = Path.cwd().parent.parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [ ]:
TAG = 'hurdle_ar1_s_p'

In [ ]:
pairs = pd.read_csv(INPUT / 'transitions.csv', dtype={'stage':str})
panel = pd.read_csv(INPUT / 'panel.csv', index_col='dblp_id')
q0 = panel['0'].dropna().to_numpy()
DEST = OUTPUT / TAG
DEST.mkdir(exist_ok=True)

In [ ]:
rows = []
for label, start, end in CONFIG['fit_stages']:
    block = pairs.loc[pairs.age.between(start, end)]
    positive = block.loc[block.state.eq(1) & block.next_state.eq(1)]
    X = np.column_stack([np.ones(len(positive)), np.log(positive.q)])
    coef = np.linalg.lstsq(X, np.log(positive.q_next), rcond=None)[0]
    residual = np.log(positive.q_next) - X @ coef
    rows.append({'stage':label, 'start':start, 'end':end, 'intercept':coef[0], 'beta':coef[1],
                 'sigma':np.sqrt(np.mean(residual**2)), 'n_positive':len(positive),
                 'p_restart':block.loc[block.state.eq(0), 'next_state'].mean()})
params = pd.DataFrame(rows)

In [ ]:
# stage intercepts and a shared nonpositive productivity effect
active = pairs.loc[pairs.state.eq(1)].copy()
labels = [label for label,_,_ in CONFIG['fit_stages']]
ends = [end for _,_,end in CONFIG['fit_stages']]
active['stage'] = pd.cut(active.age, [-1,*ends], labels=labels)
stage_ix = active.stage.map(dict(zip(labels, range(4)))).to_numpy()
x = np.log(active.q.to_numpy() + EPS)
y = active.next_state.eq(0).to_numpy(dtype=float)
initial = active.groupby('stage').next_state.apply(lambda v: 1-v.mean()).reindex(labels).clip(1e-5, 1-1e-5)
def objective(theta):
    eta = theta[:4][stage_ix] + theta[4] * x
    return np.sum(np.logaddexp(0, eta) - y * eta)
def gradient(theta):
    eta = theta[:4][stage_ix] + theta[4] * x
    error = expit(eta) - y
    return np.r_[np.bincount(stage_ix, weights=error, minlength=4), error @ x]
# bounded coefficient includes gamma=0; equivalent likelihood without exp(gamma) ill-conditioning
result = minimize(objective, np.r_[np.log(initial/(1-initial)), -0.1], jac=gradient,
                  bounds=[(None,None)]*4 + [(None,0)], method='L-BFGS-B',
                  options={'ftol':1e-12, 'gtol':1e-7, 'maxiter':2000})
params['dropout_intercept'] = result.x[:4]
params['dropout_log_q_coefficient'] = result.x[4]
optimizer = pd.DataFrame([{'success':result.success, 'nll':result.fun, 'message':str(result.message),
                           'max_gradient':np.max(np.abs(result.jac))}])
if not result.success:
    raise RuntimeError(result.message)

In [ ]:
restart = pairs.loc[pairs.state.eq(0) & pairs.next_state.eq(1), 'q_next']
initialization = pd.DataFrame([{'p_init_active':np.mean(q0>0), 'positive_scale':q0[q0>0].mean(),
                                'restart_scale':restart.mean(), 'n_restarts':len(restart)}])
params.to_csv(DEST / 'parameters.csv', index=False)
initialization.to_csv(DEST / 'initialization.csv', index=False)
optimizer.to_csv(DEST / 'optimizer.csv', index=False)
display(params)
display(optimizer)